# EXPLORACION DATOS

## 1. IMPORTAR LIBRERIAS

In [1]:
def instalar_e_importar_librerias():
    """
    Instala (si es necesario) e importa todas las librerías requeridas
    para la carga de datos, preprocesamiento y modelado predictivo.
    """
    import importlib
    import subprocess
    import sys

    librerias = [
        # Manipulación de datos
        "pandas",
        "numpy",

        # Lectura y escritura de archivos Parquet
        "pyarrow",
        "fastparquet",

        # Visualización (opcional para análisis exploratorio)
        "matplotlib",
        "seaborn",

        # Modelado y métricas
        "scikit-learn",

        # Monitoreo opcional de memoria
        "psutil"
    ]

    # Instalación automática si falta alguna librería
    for libreria in librerias:
        try:
            importlib.import_module(libreria)
        except ImportError:
            print(f"📦 Instalando {libreria}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", libreria])

    # Importar las librerías globalmente
    global pd, np, plt, sns
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    # Configuraciones opcionales
    pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
    sns.set(style="whitegrid", palette="pastel")

    print("✅ Todas las librerías instaladas e importadas correctamente.")



## 2.FUNCIONES

### 2.0. FUNCIÓN PARA LIMPIAR RAM

In [2]:
def limpiar_memoria():
    """
    Libera memoria RAM antes de ejecutar procesos pesados.
    - Elimina variables temporales.
    - Fuerza la recolección de basura.
    - Muestra el uso de memoria después de la limpieza (si psutil está disponible).
    """
    import gc
    import sys

    # Intenta limpiar variables globales del entorno (excepto las necesarias)
    global_vars = list(globals().keys())
    for var in global_vars:
        if var not in ['gc', 'sys', 'limpiar_memoria']:
            del globals()[var]

    # Forzar recolección de basura
    gc.collect()

    try:
        import psutil
        memoria = psutil.virtual_memory()
        print(f"🧹 Memoria limpiada — Uso actual: {memoria.percent}% de {round(memoria.total / 1e9, 2)} GB")
    except ImportError:
        print("🧹 Memoria limpiada (instala 'psutil' para monitorear el uso).")

In [3]:
def limpiar_memoria_segura():
    """
    Libera memoria RAM sin borrar funciones ni variables esenciales.
    Ideal para notebooks (VS Code o Jupyter).
    """
    import gc
    import psutil

    # Forzar liberación de objetos no referenciados
    gc.collect()

    # Mostrar memoria usada
    memoria = psutil.virtual_memory()
    print(f"🧹 Memoria limpiada — Uso actual: {memoria.percent}% de {round(memoria.total / 1e9, 2)} GB")

### 2.1. FUNCIÓN PARA LEER ARCHIVOS

In [4]:
def importar_parquet(ruta_archivo):
    """
    Importa un archivo Parquet desde la ruta especificada.

    Parámetros:
    -----------
    ruta_archivo : str
        Ruta completa del archivo .parquet a cargar.

    Retorna:
    --------
    DataFrame de pandas con los datos importados.
    """
    import os

    if not os.path.exists(ruta_archivo):
        raise FileNotFoundError(f"❌ No se encontró el archivo en la ruta especificada: {ruta_archivo}")

    df = pd.read_parquet(ruta_archivo)
    print(f"✅ Archivo cargado correctamente: {ruta_archivo}")
    print(f"   Filas: {len(df):,} | Columnas: {len(df.columns)}")
    return df

### 2.2.FUNCIÓN MAIN PARA EJECUTAR TODO EL CÓDIGO

In [5]:
def main():
    limpiar_memoria_segura()
    instalar_e_importar_librerias()

    ruta = r"D:\2025-15\Depliegue Soluciones Analìticas\Proyecto\Proyecto_DSA\data\processed\defunciones.parquet"
    df_defunciones = importar_parquet(ruta)

    print("\n🧾 Vista previa del DataFrame:")
    print(df_defunciones.head())

    return df_defunciones


### 2.4 FUNCIÓN PARA ANÁLISIS EXPLORATORIO DE DATOS

In [6]:
def analisis_exploratorio_datos(df, max_cols=20, sample_size=1_000_000):
    """
    Realiza un análisis exploratorio general del DataFrame optimizado para bajo consumo de memoria.
    
    - Estadísticas descriptivas
    - Distribución de los datos (nulos y no nulos)
    - Ordena columnas por cantidad de datos no nulos (de mayor a menor)
    
    Parámetros:
    -----------
    df : pandas.DataFrame
        DataFrame a analizar.
    max_cols : int
        Número máximo de columnas a mostrar por tipo (por defecto 20).
    sample_size : int
        Tamaño máximo de muestra para calcular estadísticas (por defecto 1 millón de filas).
    """
    import pandas as pd
    import numpy as np

    print("🔍 INICIO DEL ANÁLISIS EXPLORATORIO DE DATOS\n" + "="*60)

    # 1️⃣ Dimensiones y tipos
    print(f"📏 Dimensiones: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")
    print(f"🧠 Tipos de datos: {dict(df.dtypes.value_counts())}\n")

    # 2️⃣ Completitud de los datos
    print("📊 Completitud de columnas (ordenadas por datos no nulos):")
    completitud = pd.DataFrame({
        "No Nulos": df.notnull().sum(),
        "Nulos": df.isnull().sum()
    })
    completitud["% Completitud"] = 100 * completitud["No Nulos"] / len(df)
    completitud = completitud.sort_values(by="No Nulos", ascending=False)
    print(completitud.head(max_cols).round(2))

    # 3️⃣ Reducir muestra para análisis estadístico
    if len(df) > sample_size:
        df_sample = df.sample(sample_size, random_state=42)
        print(f"\n⚡ Usando muestra aleatoria de {sample_size:,} filas para estadísticas.")
    else:
        df_sample = df

    # 4️⃣ Estadísticas descriptivas numéricas
    num_cols = df_sample.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        print("\n📈 Estadísticas descriptivas numéricas:")
        stats = df_sample[num_cols].describe().T
        print(stats.head(max_cols).round(2))
    else:
        print("\n⚠️ No se encontraron columnas numéricas.")

    # 5️⃣ Variables categóricas (sin agotar RAM)
    cat_cols = df_sample.select_dtypes(exclude=[np.number]).columns
    if len(cat_cols) > 0:
        print("\n🔠 Resumen de variables categóricas:")
        resumen_cat = []
        for c in cat_cols[:max_cols]:
            vals_unicos = df_sample[c].nunique(dropna=True)
            modo = df_sample[c].mode(dropna=True)
            mas_frec = modo.iloc[0] if not modo.empty else None
            freq = df_sample[c].value_counts(dropna=True).iloc[0] if not df_sample[c].value_counts(dropna=True).empty else None
            resumen_cat.append((c, vals_unicos, mas_frec, freq))
        resumen_cat = pd.DataFrame(resumen_cat, columns=["Columna", "Valores Únicos", "Más Frecuente", "Frecuencia"])
        print(resumen_cat.to_string(index=False))
    else:
        print("\n⚠️ No se encontraron columnas categóricas.")

    # 6️⃣ Resumen final
    print("\n✅ Análisis exploratorio finalizado.")
    return completitud



### 2.5. FUNCIÓN PARA REDUCCIÓN DE DIMENSIONALIDAD

In [ ]:
def preparar_datos_y_reducir_dimensionalidad(
    df,
    n_componentes=8,
    usar_muestra=True,
    max_filas=1_000_000,
    columna_objetivo="NUM_MUERTES"
):
    """
    Prepara los datos para modelar de forma eficiente en memoria:
      - Agrupa por Departamento, Municipio, Año, Mes si no existe NUM_MUERTES
      - Limpia y convierte tipos de datos
      - Codifica variables categóricas
      - Escala y aplica PCA opcionalmente
      - Divide en train/test
    """
    import pandas as pd
    import numpy as np
    import gc
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA

    print("\n⚙️ INICIO DE PREPARACIÓN DE DATOS Y REDUCCIÓN DE DIMENSIONALIDAD")
    print("=" * 65)

    # 1️⃣ --- Control de memoria inicial ---
    memoria_inicial = df.memory_usage(deep=True).sum() / 1e6
    print(f"📦 Tamaño inicial del DataFrame: {memoria_inicial:,.2f} MB")

    if usar_muestra and len(df) > max_filas:
        df = df.sample(n=max_filas, random_state=42)
        print(f"⚡ Usando muestra de {max_filas:,} filas (de {len(df):,})")

    # 2️⃣ --- Crear NUM_MUERTES si no existe ---
    if columna_objetivo not in df.columns:
        columnas_agrupacion = ["COD_DPTO_DESC", "COD_MUNIC_DESC", "ANO", "MES"]
        columnas_agrupacion = [c for c in columnas_agrupacion if c in df.columns]

        if len(columnas_agrupacion) < 2:
            raise ValueError("❌ No se encontraron columnas suficientes para agrupar los fallecidos.")

        print(f"🧮 Agrupando datos por {columnas_agrupacion} para calcular número de muertes...")
        df = (
            df.groupby(columnas_agrupacion, as_index=False)
              .size()
              .rename(columns={"size": columna_objetivo})
        )
        print(f"✅ Columna '{columna_objetivo}' creada correctamente (agregada por {columnas_agrupacion})")

    # 3️⃣ --- Limpieza de columnas poco informativas ---
    df = df.loc[:, df.nunique() > 1]
    df = df.loc[:, df.isnull().mean() < 0.95]

    # 4️⃣ --- Separar variables ---
    y = df[columna_objetivo]
    X = df.drop(columns=[columna_objetivo])

    # 5️⃣ --- Conversión de tipos (más eficiente) ---
    for col in X.select_dtypes(include=["object", "category"]).columns:
        X[col] = X[col].astype("category").cat.codes

    # 6️⃣ --- Rellenar nulos ---
    X = X.fillna(0)

    # 7️⃣ --- Estandarizar ---
    scaler = StandardScaler(copy=False)
    X_scaled = scaler.fit_transform(X.astype(np.float32))

    # 8️⃣ --- PCA (opcional) ---
    if n_componentes is not None and n_componentes > 0:
        n_componentes = min(n_componentes, X_scaled.shape[1])
        print(f"🔹 Aplicando PCA con {n_componentes} componentes...")
        pca = PCA(n_components=n_componentes, svd_solver='randomized', random_state=42)
        X_reduced = pca.fit_transform(X_scaled)
        varianza = pca.explained_variance_ratio_.sum() * 100
        print(f"✅ PCA completado — Varianza explicada: {varianza:.2f}%")
    else:
        X_reduced = X_scaled
        print("⚠️ PCA omitido, se mantienen todas las variables numéricas.")

    # 9️⃣ --- Train/test ---
    X_train, X_test, y_train, y_test = train_test_split(
        X_reduced, y.values, test_size=0.2, random_state=42
    )

    # 🔟 --- Liberar memoria ---
    del X, X_scaled, X_reduced, y, df
    gc.collect()

    print(f"✅ Datos listos para modelar:")
    print(f"   • Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,}")
    print(f"   • Dimensiones finales: {X_train.shape[1]} variables\n")

    return X_train, X_test, y_train, y_test


### 2.6. FUNCIÓN MODELO 1: ÁRBOL DE DECISIÓN

In [8]:
def modelo_arbol_decision(X_train, y_train, X_test, max_depth=5):
    """
    Entrena un Árbol de Decisión para predecir el número de muertes.
    """
    from sklearn.tree import DecisionTreeRegressor

    model = DecisionTreeRegressor(max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return preds, model


### 2.7. FUNCIÓN MODELO 2: RANDOM FOREST

In [9]:
def modelo_random_forest(X_train, y_train, X_test, n_estimators=100):
    """
    Entrena un Random Forest para predecir el número de muertes.
    """
    from sklearn.ensemble import RandomForestRegressor

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return preds, model


### 2.8. FUNCIÓN MODELO 3: REGRESIÓN LINEAL

In [10]:
def modelo_regresion_lineal(X_train, y_train, X_test):
    """
    Entrena una Regresión Lineal para predecir el número de muertes.
    """
    from sklearn.linear_model import LinearRegression

    model = LinearRegression()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return preds, model

### 2.9. FUNCIÓN PARA EVALUAR MODELOS

In [11]:
def evaluar_modelos(y_test, preds_dict):
    """
    Evalúa el desempeño de varios modelos usando RMSE, MAE y R2.
    
    Parámetros:
    -----------
    y_test : Series
        Valores reales del conjunto de prueba
    preds_dict : dict
        Diccionario con nombre del modelo y sus predicciones
        Ejemplo: {"Árbol": preds1, "RandomForest": preds2}
    """
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    import numpy as np
    import pandas as pd

    resultados = []
    for nombre, preds in preds_dict.items():
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        r2 = r2_score(y_test, preds)
        resultados.append([nombre, rmse, mae, r2])

    resultados_df = pd.DataFrame(resultados, columns=["Modelo", "RMSE", "MAE", "R2"])
    print("\n📊 RESULTADOS DE EVALUACIÓN:")
    display(resultados_df.sort_values("R2", ascending=False))
    return resultados_df


## 3.EJECUCIÓN

### 3.1. EJECUTAR MAIN

In [12]:
df = main()

🧹 Memoria limpiada — Uso actual: 31.2% de 34.24 GB
📦 Instalando scikit-learn...
✅ Todas las librerías instaladas e importadas correctamente.
✅ Archivo cargado correctamente: D:\2025-15\Depliegue Soluciones Analìticas\Proyecto\Proyecto_DSA\data\processed\defunciones.parquet
   Filas: 8,825,031 | Columnas: 137

🧾 Vista previa del DataFrame:
    ANO AREA_RES AREA_RES_DESC ASIS_MED ASIS_MED_DESC A_DEFUN  \
0  1979                                                     1   
1  1979                                                     1   
2  1979                                                     1   
3  1979                                                     1   
4  1979                                                     1   

         A_DEFUN_DESC CAUSA_666 CAUSA_666_DESC CAUSA_667  ... TIPO_EMB  \
0  CABECERA MUNICIPAL                                     ...            
1  CABECERA MUNICIPAL                                     ...            
2  CABECERA MUNICIPAL                         

### 3.2. EJECUTAR ANÁLISIS EXPLORATORIO DE DATOS

In [13]:
analisis_exploratorio_datos(df)

🔍 INICIO DEL ANÁLISIS EXPLORATORIO DE DATOS
📏 Dimensiones: 8,825,031 filas x 137 columnas
🧠 Tipos de datos: {dtype('O'): 137}

📊 Completitud de columnas (ordenadas por datos no nulos):
                                No Nulos  Nulos  % Completitud
ANO                              8825031      0        100.000
NOMBRE_DEPARTAMENTO_OCURRENCIA   8825031      0        100.000
MU_PARTO_DESC                    8825031      0        100.000
NIVEL_EDU                        8825031      0        100.000
NIVEL_EDU_DESC                   8825031      0        100.000
NIV_EDUM                         8825031      0        100.000
NIV_EDUM_DESC                    8825031      0        100.000
NOMBRE_DEPARTAMENTO              8825031      0        100.000
NOMBRE_DEPARTAMENTO_RESIDENCIA   8825031      0        100.000
GRU_ED1_DESC                     8825031      0        100.000
NOMBRE_MUNICIPIO                 8825031      0        100.000
NOMBRE_MUNICIPIO_OCURRENCIA      8825031      0        100.

,No Nulos,Nulos,% Completitud
ANO,8825031,0,100.000
NOMBRE_DEPARTAMENTO_OCURRENCIA,8825031,0,100.000
MU_PARTO_DESC,8825031,0,100.000
NIVEL_EDU,8825031,0,100.000
NIVEL_EDU_DESC,8825031,0,100.000
...,...,...,...
C_DIR1,8825031,0,100.000
C_BAS1_DESC,8825031,0,100.000
C_BAS1,8825031,0,100.000
C_ANT3_DESC,8825031,0,100.000


In [14]:
display(df.head())

,ANO,AREA_RES,AREA_RES_DESC,ASIS_MED,ASIS_MED_DESC,A_DEFUN,A_DEFUN_DESC,CAUSA_666,CAUSA_666_DESC,CAUSA_667,...,TIPO_EMB,TIPO_EMB_DESC,T_GES,T_GES_AGRU_CIE,T_GES_AGRU_CIE_DESC,T_GES_DESC,T_PARTO,T_PARTO_DESC,ULTCURFAL,ULTCURMAD
0,1979,,,,,1,CABECERA MUNICIPAL,,,,...,,,,,,,,,,
1,1979,,,,,1,CABECERA MUNICIPAL,,,,...,,,,,,,,,,
2,1979,,,,,1,CABECERA MUNICIPAL,,,,...,,,,,,,,,,
3,1979,,,,,1,CABECERA MUNICIPAL,,,,...,,,,,,,,,,
4,1979,,,,,1,CABECERA MUNICIPAL,,,,...,,,,,,,,,,


### 3.3. EJECUTAR PREPARACIÓN DE DATOS Y MODELOS

In [15]:
# 1️⃣ Preparar datos
X_train, X_test, y_train, y_test = preparar_datos_y_reducir_dimensionalidad(
    df,
    n_componentes=8,
    usar_muestra=True,
    max_filas=1_000_000
)

# 2️⃣ Entrenar modelos
pred_arbol, modelo1 = modelo_arbol_decision(X_train, y_train, X_test)
pred_rf, modelo2 = modelo_random_forest(X_train, y_train, X_test)
pred_lr, modelo3 = modelo_regresion_lineal(X_train, y_train, X_test)

# 3️⃣ Evaluar
resultados = evaluar_modelos(y_test, {
    "Árbol de Decisión": pred_arbol,
    "Random Forest": pred_rf,
    "Regresión Lineal": pred_lr
})



⚙️ INICIO DE PREPARACIÓN DE DATOS Y REDUCCIÓN DE DIMENSIONALIDAD
📦 Tamaño inicial del DataFrame: 64,875.54 MB
⚡ Usando muestra de 1,000,000 filas (de 1,000,000)


ValueError: ❌ No se encontró la columna objetivo 'NUM_MUERTES' en el DataFrame.